In [1]:
from google.colab import drive
drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [ ]:
import sqlite3
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

# Optional: For pretty scientific plots
try:
    import scienceplots
    plt.style.use(["science", "grid", "high-vis", "no-latex"])
except ImportError:
    plt.style.use("default")

Imports and Setup

In [ ]:
import os
print(os.listdir('/content/drive/MyDrive/AML_Semantic_Segmentation/PIDNet/datasets/'))

['__pycache__', 'base_dataset.py', 'readme.txt', 'camvid.py', '__init__.py', 'cityscapes.py', 'loveda.py']


Instantiate the Dataset

In [ ]:
# Set these paths to your actual data locations
root = '/content/drive/MyDrive/AML_Semantic_Segmentation/data/LoveDA/Train/Urban/images_png'
list_path = '/content/drive/MyDrive/AML_Semantic_Segmentation/PIDNet/data/list/loveda/train.lst'

train_dataset = LoveDA(
    root=root,
    list_path=list_path,
    num_classes=7,
    multi_scale=False,
    flip=False,
    ignore_label=255,
    base_size=1024,
    crop_size=(1024, 1024)
)

print(f"Train Dataset length: {len(train_dataset)}")

# Set these paths to your actual data locations
root = '/content/drive/MyDrive/AML_Semantic_Segmentation/data/LoveDA/Val/Urban/images_png'
list_path = '/content/drive/MyDrive/AML_Semantic_Segmentation/PIDNet/data/list/loveda/val.lst'

val_dataset = LoveDA(
    root=root,
    list_path=list_path,
    num_classes=7,
    multi_scale=False,
    flip=False,
    ignore_label=255,
    base_size=1024,
    crop_size=(1024, 1024)
)

print(f"Val Dataset length: {len(val_dataset)}")

NameError: name 'LoveDA' is not defined

Visualize a Few Samples and Print Label Ranges

In [ ]:
import matplotlib.pyplot as plt
import numpy as np

sys.path.append('/content/drive/MyDrive/AML_Semantic_Segmentation/PIDNet/tools')  # so you can import datasets
sys.path.append('/content/drive/MyDrive/AML_Semantic_Segmentation/PIDNet/')
from visualization import decode_segmap

# Use your dataset's mean and std (from config or dataset)
mean = np.array([0.485, 0.456, 0.406])
std = np.array([0.229, 0.224, 0.225])

num_samples = 2
all_labels = []

for idx in range(num_samples):
    image, label, edge, size, name = train_dataset[idx]
    # image: torch.Tensor (C, H, W) normalized
    # label: np.ndarray or torch.Tensor (H, W)

    # Unnormalize image for visualization
    if isinstance(image, torch.Tensor):
        img_np = image.numpy().transpose(1,2,0)  # (H, W, C)
    else:
        img_np = image.transpose(1,2,0)
    img_np = (img_np * std + mean)
    img_np = np.clip(img_np, 0, 1)

    # Visualize the label mask with color
    label_vis = decode_segmap(label, ignore_index=-1)  # or whatever your ignore index is

    plt.figure(figsize=(12,6))
    plt.subplot(1,2,1)
    plt.imshow(img_np)
    plt.title(f"Original Image: {name}")
    plt.axis('off')

    plt.subplot(1,2,2)
    plt.imshow(label_vis)
    plt.title(f"Label visualization for {name}")
    plt.axis('off')
    plt.show()

    # See unique label values
    print("Unique label values in this mask:", np.unique(label))
    all_labels.append(label)

# Flatten all labels and print global min/max/unique
all_labels_flat = np.concatenate([l.flatten() for l in all_labels])
print(f"\nGlobal label value range in {num_samples} samples: {all_labels_flat.min()} to {all_labels_flat.max()}")
print(f"Unique label values in {num_samples} samples: {np.unique(all_labels_flat)}")


NameError: name 'sys' is not defined

Histogram of Label Distribution

In [ ]:
plt.figure(figsize=(8,4))
plt.hist(all_labels_flat, bins=np.arange(-1,8)-0.5, rwidth=0.8)
plt.title("Label Value Distribution in Sampled Masks")
plt.xlabel("Label Value")
plt.ylabel("Pixel Count")
plt.xticks(np.arange(0, 7))
plt.show()

In [ ]:
import numpy as np
from tqdm import tqdm
from loveda import LoveDA  # adjust import if needed

# Instantiate your dataset (adjust paths/config as needed)
dataset = LoveDA(
    root='data/',  # or your actual root
    list_path='/content/drive/MyDrive/AML_Semantic_Segmentation/PIDNet/data/list/loveda/val.lst',  # or your actual list
    num_classes=7,
    multi_scale=False,
    flip=False,
    ignore_label=255,
    base_size=720,
    crop_size=(512, 512)
)

all_labels = []

for i in tqdm(range(len(dataset)), desc="Scanning dataset"):
    _, label, _, _, _ = dataset[i]
    all_labels.append(label.flatten())

all_labels = np.concatenate(all_labels)
unique_labels = np.unique(all_labels)

print(f"Unique label values in the dataset: {unique_labels}")
print(f"Label value range: min={unique_labels.min()}, max={unique_labels.max()}")

import collections
counts = collections.Counter(all_labels)
print("Counts per label:", dict(counts))

# RUN

Install Dependencies

In [5]:
!pip install tensorboardX thop albumentations tqdm pyyaml yacs

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 87.2/87.2 kB 2.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 363.4/363.4 MB 3.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 13.8/13.8 MB 112.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 24.6/24.6 MB 91.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 883.7/883.7 kB 58.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 664.8/664.8 MB 1.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 211.5/211.5 MB 11.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 56.3/56.3 MB 41.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 127.9/127.9 MB 19.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 207.5/207.5 MB 3.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 188.7/188.7 MB 13.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 21.1/21.1 MB 102.2 MB/s eta 0:00:00
  Attempting uninstall: nvidia-nvjitlink

Generate dataset list files

In [3]:
%cd /content/drive/MyDrive/AML_Semantic_Segmentation/PIDNet

!python -m datasets.loveda

/content/drive/MyDrive/AML_Semantic_Segmentation/PIDNet
<frozen runpy>:128: RuntimeWarning: 'datasets.loveda' found in sys.modules after import of package 'datasets', but prior to execution of 'datasets.loveda'; this may result in unpredictable behaviour


In [ ]:
import sys

sys.path.append('/content/drive/MyDrive/AML_Semantic_Segmentation/PIDNet/')

sys.path.append('/content/drive/MyDrive/AML_Semantic_Segmentation/PIDNet/datasets/')

sys.path.append('/content/drive/MyDrive/AML_Semantic_Segmentation/PIDNet/tools/')

Train PIDNet-S on

In [6]:
%cd /content/drive/MyDrive/AML_Semantic_Segmentation/PIDNet
!python ./tools/train.py --cfg ./configs/loveda/pidnet_loveda_urban.yaml

/content/drive/MyDrive/AML_Semantic_Segmentation/PIDNet
Seeding with 304
=> creating output/loveda/pidnet_loveda_urban
=> creating log/loveda/pidnet_small/pidnet_loveda_urban_2025-08-16-21-12
Namespace(cfg='./configs/loveda/pidnet_loveda_urban.yaml', seed=304, opts=[])
AUTO_RESUME: False
CUDNN:
  BENCHMARK: True
  DETERMINISTIC: False
  ENABLED: True
DATASET:
  DATASET: loveda
  EXTRA_TRAIN_SET: 
  NUM_CLASSES: 7
  ROOT: data/
  TEST_SET: list/loveda/val.lst
  TRAIN_SET: list/loveda/train.lst
GPUS: (0,)
LOG_DIR: log
LOSS:
  BALANCE_WEIGHTS: [0.4, 1.0]
  CLASS_BALANCE: False
  OHEMKEEP: 131072
  OHEMTHRES: 0.9
  SB_WEIGHTS: 1.0
  USE_OHEM: True
MODEL:
  ALIGN_CORNERS: True
  NAME: pidnet_small
  NUM_OUTPUTS: 2
  PRETRAINED: ./pretrained_models/imagenet/PIDNet_S_ImageNet.pth.tar
OUTPUT_DIR: output
PIN_MEMORY: True
PRINT_FREQ: 10
TEST:
  BASE_SIZE: 1024
  BATCH_SIZE_PER_GPU: 6
  FLIP_TEST: False
  IMAGE_SIZE: [1024, 1024]
  MODEL_FILE: 
  MULTI_SCALE: False
  OUTPUT_INDEX: 1
TRAIN:
  BASE

Evaluate on LoveDA-urban

In [2]:
%cd /content/drive/MyDrive/AML_Semantic_Segmentation/PIDNet
!python tools/eval.py --cfg configs/loveda/pidnet_loveda_urban.yaml

/content/drive/MyDrive/AML_Semantic_Segmentation/PIDNet
=> creating output/loveda/pidnet_loveda_urban
=> creating log/loveda/pidnet_small/pidnet_loveda_urban_2025-08-16-22-15
Namespace(cfg='configs/loveda/pidnet_loveda_urban.yaml', opts=[])
CfgNode({'OUTPUT_DIR': 'output', 'LOG_DIR': 'log', 'GPUS': (0,), 'WORKERS': 6, 'PRINT_FREQ': 10, 'AUTO_RESUME': False, 'PIN_MEMORY': True, 'CUDNN': CfgNode({'BENCHMARK': True, 'DETERMINISTIC': False, 'ENABLED': True}), 'MODEL': CfgNode({'NAME': 'pidnet_small', 'PRETRAINED': './pretrained_models/imagenet/PIDNet_S_ImageNet.pth.tar', 'ALIGN_CORNERS': True, 'NUM_OUTPUTS': 2}), 'LOSS': CfgNode({'USE_OHEM': True, 'OHEMTHRES': 0.9, 'OHEMKEEP': 131072, 'CLASS_BALANCE': False, 'BALANCE_WEIGHTS': [0.4, 1.0], 'SB_WEIGHTS': 1.0}), 'DATASET': CfgNode({'ROOT': 'data/', 'DATASET': 'loveda', 'NUM_CLASSES': 7, 'TRAIN_SET': 'list/loveda/train.lst', 'EXTRA_TRAIN_SET': '', 'TEST_SET': 'list/loveda/val.lst'}), 'TRAIN': CfgNode({'IMAGE_SIZE': [1024, 1024], 'BASE_SIZE': 1

Visualize preidctions

In [3]:
!python tools/visualization.py --config configs/loveda/pidnet_loveda_urban.yaml --checkpoint output/best.pt --split val --output_dir visualizations --num_samples 10

Traceback (most recent call last):
  File "/content/drive/MyDrive/AML_Semantic_Segmentation/PIDNet/tools/visualization.py", line 112, in <module>
    visualize(args.config, args.checkpoint, args.split, args.output_dir, args.num_samples)
  File "/content/drive/MyDrive/AML_Semantic_Segmentation/PIDNet/tools/visualization.py", line 39, in visualize
    import models
ModuleNotFoundError: No module named 'models'


Show some visualizations

In [ ]:
import matplotlib.pyplot as plt
import glob
from PIL import Image

vis_dir = 'visualizations'
imgs = sorted(glob.glob(f"{vis_dir}/*.png"))
for img_path in imgs[:5]:
    img = Image.open(img_path)
    plt.figure(figsize=(10,5))
    plt.imshow(img)
    plt.axis('off')
    plt.show()